# Gradient Cobra

## Combine Classifier

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from cobra.combine_classifier import CombineClassifier

model = CombineClassifier(
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
preds = model.predict(X_test)
accuracy = (preds == y_test).mean()
print(f"Test set accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

baselines = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "logistic_regression": LogisticRegression(max_iter=5000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(),
    "knn": KNeighborsClassifier(),
    "svm": SVC()
}

results = []

for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results.append(("baseline_" + name, acc))

model = CombineClassifier(
    estimators=baselines.keys(),
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
results.append(("combine_classifier", acc))

for name, acc in results:
    print(f"{name}: {acc:.4f}")


In [ ]:
from cobra.core.estimators.base import BaseEstimator, EstimatorFactory
EstimatorFactory.available()

# GradientCobra

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [2]:
from cobra.gradientcobra import GradientCOBRA
import numpy as np

model = GradientCOBRA(
    splitter="kfold",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    optimizer="grad",
    random_state=42
)
model.fit(X_train, y_train)

len folds: 5


Gradient Descent: 100%|██████████| 100/100 [03:03<00:00,  1.83s/it, best=0.5155, grad=0.3365, score=0.5155]


,estimators,None
,estimators_params,None
,distance,'euclidean'
,distance_params,None
,kernel,'rbf'
,kernel_params,None
,aggregator,'weighted_mean'
,aggregator_params,None
,splitter,'kfold'
,splitter_params,None
,loss,'mse'


In [6]:
model.optimization_outputs_['histories']


[{'iteration': 0,
  'x': array([0.50762719]),
  'score': 0.7777993388544715,
  'best_score': 0.7777993388544715,
  'gradient': array([-0.76271864])},
 {'iteration': 1,
  'x': array([0.51517649]),
  'score': 0.7721291129071748,
  'best_score': 0.7721291129071748,
  'gradient': array([-0.75493024])},
 {'iteration': 2,
  'x': array([0.52264911]),
  'score': 0.7665733536299381,
  'best_score': 0.7665733536299381,
  'gradient': array([-0.74726174])},
 {'iteration': 3,
  'x': array([0.5300462]),
  'score': 0.7611291968447595,
  'best_score': 0.7611291968447595,
  'gradient': array([-0.73970918])},
 {'iteration': 4,
  'x': array([0.53736889]),
  'score': 0.7557938831097432,
  'best_score': 0.7557938831097432,
  'gradient': array([-0.73226888])},
 {'iteration': 5,
  'x': array([0.54461826]),
  'score': 0.7505647510101998,
  'best_score': 0.7505647510101998,
  'gradient': array([-0.72493741])},
 {'iteration': 6,
  'x': array([0.55179538]),
  'score': 0.7454392310000291,
  'best_score': 0.745439

In [8]:
import numpy as np
preds = model.predict(X_test)
mse = np.mean((preds - y_test) ** 2)
print(f"Test set MSE: {mse:.4f}")

Test set MSE: 0.5305


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

baselines = {
    "linear": LinearRegression(),
    "ridge": Ridge(),
    "random_forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "svm": SVR()
}
results = []
for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    results.append(("baseline_" + name, mse))

model = GradientCOBRA(
    estimators=baselines.keys(),
    splitter="holdout",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
results.append(("gradient_cobra", mse))

for name, mse in results:
    print(f"{name}: {mse:.4f}")

# MixCobra

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [ ]:
from gradientcobra.mixcobra import MixCOBRARegressor as OldMixCOBRARegressor
from sklearn.metrics import mean_squared_error

model = OldMixCOBRARegressor()
model.fit(X_train, y_train)

In [ ]:
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
print(f"Test set MSE: {mse:.4f}")

In [2]:
from cobra.mixcobra import MixCOBRARegressor
model = MixCOBRARegressor(
    splitter="kfold",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    loss="mse",
    optimizer="grad",
    random_state=42
)
model.fit(X_train, y_train)

Gradient Descent: 100%|██████████| 100/100 [09:03<00:00,  5.43s/it, best=1.1184, grad=0.1832, score=1.1184]


TypeError: MixCOBRARegressor._optimize_hyperparameters.<locals>.objective() takes 1 positional argument but 2 were given

In [ ]:
model.optimization_outputs_

In [ ]:
from sklearn.metrics import mean_squared_error
preds = model.predict(X_test, pred_X=X_test)
mse = mean_squared_error(y_test, preds)
print(f"Test set MSE: {mse:.4f}")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

from cobra.mixcobra import MixCOBRARegressor

# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}

# train single models
for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = mean_squared_error(y_test, pred)

# ======================
# MIXCOBRA
# ======================
mix = MixCOBRARegressor(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    alpha_grid=np.linspace(0.1, 3.0, 10),
    beta_grid=np.linspace(0.1, 3.0, 10),
    random_state=42
)

mix.fit(X_train, y_train)
mix_pred = mix.predict(X_test)

results["MixCOBRA"] = mean_squared_error(y_test, mix_pred)

# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")